# Stacking Ensemble for Heart Disease (OOF-based)

This notebook builds a stacking ensemble using out-of-fold (OOF) predictions from multiple base models.

**Inputs**
- `train.csv`, `test.csv`, `sample_submission.csv` (Kaggle input or local `data/raw`)
- `data/submissions/oof_*.csv` and `data/submissions/pred_*.csv`

**Outputs**
- `data/submissions/submission_stacking.csv`

**TODO**
- Run the base model notebooks to generate OOF and test predictions and save them as:
  - `data/submissions/oof_027.csv` + `data/submissions/pred_027.csv`
  - `data/submissions/oof_031.csv` + `data/submissions/pred_031.csv`
  - `data/submissions/oof_033.csv` + `data/submissions/pred_033.csv`
  - `data/submissions/oof_035.csv` + `data/submissions/pred_035.csv`

Notebooks to run (as needed):
- `nb/027_real_hyperparameter_with_early_stopping.ipynb`
- `nb/031_real_hyperparameter_seed_bugging.ipynb`
- `nb/033_realmlp-numeric-encoding.ipynb`
- `nb/035-label-flipping.ipynb`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score


## Config

In [ ]:
MODEL_TAGS = ["027", "031", "033", "035"]
PRED_DIR = Path("data/submissions")
N_SPLITS = 5
RANDOM_STATE = 42
OUTPUT_PATH = PRED_DIR / "submission_stacking.csv"


## Data Loading (Kaggle or local)

In [ ]:
def resolve_competition_data_dir() -> Path:
    kaggle_dir = Path("/kaggle/input/playground-series-s6e2")
    if kaggle_dir.exists():
        return kaggle_dir

    # Infer project root and use data/raw
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / "data" / "raw" / "train.csv").exists():
            return p / "data" / "raw"

    raise FileNotFoundError("Could not find competition data.")


DATA_DIR = resolve_competition_data_dir()
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train", train.shape, "test", test.shape, "sub", sub.shape)

# Detect target column
target_col = None
for cand in ["Heart Disease", "HeartDisease", "target", "label"]:
    if cand in train.columns:
        target_col = cand
        break

if target_col is None:
    raise ValueError("Target column not found in train.csv. Tried: Heart Disease, HeartDisease, target, label")

print("Detected target_col:", target_col)


## Load OOF and Test Predictions

In [ ]:
def load_prediction_csv(path: Path, expected_col: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing file: {path}.\n"
            f"TODO: Generate OOF/test predictions and save as {path.name}"
        )

    df = pd.read_csv(path)
    if "id" not in df.columns:
        raise ValueError(f"{path} missing required 'id' column")

    # If expected column is missing, try to infer a single prediction column
    if expected_col not in df.columns:
        pred_cols = [c for c in df.columns if c != "id"]
        if len(pred_cols) != 1:
            raise ValueError(
                f"{path} should have columns ['id', '{expected_col}'] or exactly one prediction column."
            )
        df = df.rename(columns={pred_cols[0]: expected_col})

    if df["id"].duplicated().any():
        raise ValueError(f"Duplicate ids found in {path}")

    return df[["id", expected_col]].copy()


def load_base_oof_and_test(model_tags: list[str], pred_dir: Path):
    oof_dfs = []
    pred_dfs = []
    for tag in model_tags:
        oof_path = pred_dir / f"oof_{tag}.csv"
        pred_path = pred_dir / f"pred_{tag}.csv"

        oof_col = f"oof_{tag}"
        pred_col = f"pred_{tag}"

        oof_df = load_prediction_csv(oof_path, oof_col)
        pred_df = load_prediction_csv(pred_path, pred_col)

        oof_dfs.append(oof_df)
        pred_dfs.append(pred_df)

    return oof_dfs, pred_dfs


PRED_DIR.mkdir(parents=True, exist_ok=True)
oof_dfs, pred_dfs = load_base_oof_and_test(MODEL_TAGS, PRED_DIR)
print(f"Loaded {len(oof_dfs)} OOF files and {len(pred_dfs)} test prediction files.")


## Build Meta Train/Test

In [ ]:
def build_meta_train_test(train_df, test_df, oof_dfs, pred_dfs, target_col):
    meta_train = train_df[["id", target_col]].copy()
    for df in oof_dfs:
        meta_train = meta_train.merge(df, on="id", how="inner")

    meta_test = test_df[["id"]].copy()
    for df in pred_dfs:
        meta_test = meta_test.merge(df, on="id", how="inner")

    if len(meta_train) != len(train_df):
        raise ValueError(
            f"Meta-train row count {len(meta_train)} != train row count {len(train_df)}."
        )
    if len(meta_test) != len(test_df):
        raise ValueError(
            f"Meta-test row count {len(meta_test)} != test row count {len(test_df)}."
        )

    if meta_train.isna().any().any():
        raise ValueError("NaNs found in meta_train. Check OOF files for missing ids.")
    if meta_test.isna().any().any():
        raise ValueError("NaNs found in meta_test. Check pred files for missing ids.")

    return meta_train, meta_test


meta_train, meta_test = build_meta_train_test(train, test, oof_dfs, pred_dfs, target_col)
print("meta_train", meta_train.shape, "meta_test", meta_test.shape)


## Baseline AUC per Base Model (OOF)

In [ ]:
y = meta_train[target_col].values
feature_cols = [c for c in meta_train.columns if c.startswith("oof_")]

for col in feature_cols:
    auc = roc_auc_score(y, meta_train[col].values)
    print(f"{col}: AUC={auc:.6f}")


## Train Meta-Model with CV

In [ ]:
X = meta_train[feature_cols].values

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

meta_oof = np.zeros(len(meta_train))
meta_test_preds = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = LogisticRegression(solver="lbfgs", max_iter=1000)
    model.fit(X_tr, y_tr)

    va_pred = model.predict_proba(X_va)[:, 1]
    meta_oof[va_idx] = va_pred

    test_pred = model.predict_proba(meta_test[[c.replace('oof_', 'pred_') for c in feature_cols]].values)[:, 1]
    meta_test_preds.append(test_pred)

    fold_auc = roc_auc_score(y_va, va_pred)
    print(f"Fold {fold}: AUC={fold_auc:.6f}")

meta_auc = roc_auc_score(y, meta_oof)
print(f"Meta-model OOF AUC: {meta_auc:.6f}")


## Generate Submission

In [ ]:
final_test_pred = np.mean(meta_test_preds, axis=0)

submission = pd.DataFrame({
    "id": meta_test["id"],
    target_col: final_test_pred,
})

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Saved submission to: {OUTPUT_PATH}")
submission.head()
